# CREAMOS UNA RED NEURONAL CON PYTORCH PARA PREDECIR EL PESO DE UN PINGUINO

# IMPORTAMOS LIBRERIAS

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# CARGAMOS EL DATASET

In [2]:
penguins = sns.load_dataset("penguins").dropna()
penguins

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,Male
...,...,...,...,...,...,...,...
338,Gentoo,Biscoe,47.2,13.7,214.0,4925.0,Female
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


# DEFINIMOS VARIABLES X Y Y

In [3]:
features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]
target = "body_mass_g"
X = penguins[features].values
y = penguins[target].values.reshape(-1, 1)

# ESCALAMOS LOS DATOS

In [4]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X = scaler_X.fit_transform(X)
y = scaler_y.fit_transform(y)

# DIVIDIR DATASET EN ENTRENAMIENTO Y PRUEBA

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convertimos las variables a tensores de Pytorch

In [7]:
X_train, X_test = torch.tensor(X_train, dtype=torch.float32), torch.tensor(X_test, dtype=torch.float32)

In [8]:
y_train, y_test = torch.tensor(y_train, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)

# CREAMOS RED NEURONAL CON PYTORCH

In [9]:
class Net(nn.Module):
  def __init__(self):
    super(Net, self).__init__()
    self.fc1 = nn.Linear(3,16)
    self.fc2 = nn.Linear(16,8)
    self.fc3 = nn.Linear(8,1)
    self.relu = nn.ReLU()

  def forward(self,x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# CONFIGURAMOS MODELO , FUNCIÓN DE PERDIDA Y OPTIMIZADOR

In [10]:
model = Net()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# ENTRENAMOS NUESTRO MODELO DE RED NEURONAL

In [11]:
epochs = 500
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if(epoch +1) % 50 == 0:
      print(f'epoch [{epoch+1}/{epochs}], Loss : {loss.item():.4f}')

epoch [50/500], Loss : 0.1778
epoch [100/500], Loss : 0.1591
epoch [150/500], Loss : 0.1480
epoch [200/500], Loss : 0.1409
epoch [250/500], Loss : 0.1365
epoch [300/500], Loss : 0.1325
epoch [350/500], Loss : 0.1271
epoch [400/500], Loss : 0.1220
epoch [450/500], Loss : 0.1179
epoch [500/500], Loss : 0.1138


# EVALUAMOS EL MODELO

In [12]:
model.eval()
y_pred = model(X_test).detach().numpy()
y_pred = scaler_y.inverse_transform(y_pred)
y_test = scaler_y.inverse_transform(y_test.numpy())

for i in range(5):
  print(f'valor predicho {y_pred[i][0]:-2f} - Valor Real : {y_test[i][0]:.2f}')

valor predicho 3357.831055 - Valor Real : 3250.00
valor predicho 4915.778320 - Valor Real : 4875.00
valor predicho 4197.384766 - Valor Real : 4000.00
valor predicho 3722.990479 - Valor Real : 3675.00
valor predicho 3860.575928 - Valor Real : 4050.00


# Evaluar con métricas de sklearn

In [13]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"R2 Score: {r2:.2f}")

MAE: 249.89
MSE: 106164.28
R2 Score: 0.83
